# Week 2 — GradCAM + LIME (Person 1) v3 — FIXED

**Run cells 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 in order, top to bottom, on a fresh runtime.**

### What changed from v2 (this pass)
1. **Import bug fixed** — Cell 1 now calls `importlib.invalidate_caches()` after adding the repo to `sys.path`, and verifies the `xai` module actually resolves *before* moving on, instead of failing three cells later with a confusing `ModuleNotFoundError`.
2. **EfficientNet-B4 GradCAM layer fixed** — `GRADCAM_LAYER['efficientnetb4']` changed from `'conv_head'` (a raw conv *before* its batchnorm — this is what caused the flat heatmap with a hot corner pixel) to `'blocks.6'` (the last MBConv stage, BN-calibrated). Verified directly against `timm`'s `efficientnet_b4` module tree.
3. **`load_image()` moved into Cell 5 (Helper Functions)** — it was only ever defined in scratch cells that aren't part of the official 1→8 run order, so a clean run from the top would have hit a `NameError` in Cell 6/7. Now it's where it belongs.
4. **Checkpoint loading made Drive-duplicate-proof** — `resolve_checkpoint()` in Cell 3 auto-finds the right `.pth` file even when Drive has appended `(1)`, `(2)`, etc. from a duplicate upload, and warns you if more than one candidate exists.
5. **The destructive "delete all .npy files" step is no longer part of the normal run** — it's what wiped the ResNet-50 results last time. It now lives in its own clearly-marked, **off-by-default** cell (4b) that you have to deliberately opt into.
6. **Removed ~14 leftover scratch/debugging cells** (repeated Drive mounts, one-off path checks, duplicate `load_image`/`CKPT` definitions) so the notebook matches the "run 1→8" instructions again.

### Before running
- Runtime → Restart runtime (clears any stale state from previous failed import attempts)
- Runtime → Change runtime type → T4 GPU


In [ ]:
# Cell 1: Mount Drive + Clone/Pull Repo + make the import reliable
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, sys, importlib, importlib.util

REPO_URL = 'https://github.com/rao-shreyashree/diabetic-retinopathy-xai.git'
REPO_DIR = '/content/diabetic-retinopathy-xai'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull')

# Only create __init__.py if missing — never overwrite an existing one
for folder in ['datasets', 'xai', 'utils']:
    init_path = os.path.join(REPO_DIR, folder, '__init__.py')
    if not os.path.exists(init_path):
        open(init_path, 'w').close()

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# THE FIX: force Python to re-scan sys.path directories. Without this,
# if 'xai' / 'datasets' / 'utils' were looked up even once before the repo
# fully existed at REPO_DIR, Python caches that empty listing — and
# `import xai` keeps failing with ModuleNotFoundError even though the
# files are sitting right there on disk.
importlib.invalidate_caches()

# Clear any partially-failed import attempts left over from earlier in
# this session (a half-imported package can also block retries).
for mod in list(sys.modules.keys()):
    if any(mod.startswith(x) for x in ['utils', 'datasets', 'xai']):
        del sys.modules[mod]

# Verify right here — fail loudly now, not three cells later.
spec = importlib.util.find_spec('xai')
if spec is None:
    raise ImportError(
        "Still can't find module 'xai' after invalidate_caches(). "
        "Go to Runtime -> Restart runtime, then run THIS cell first "
        "(before anything else touches sys.path), then re-run from the top."
    )

print(f'Repo ready at {REPO_DIR}')
print('xai module found at:', spec.origin)


In [ ]:
# Cell 2: Install Dependencies
os.system('pip install timm lime scikit-image -q')
print('Done')


In [ ]:
# Cell 3: Imports + Paths
import os, sys, json, csv, glob
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image

REPO_DIR = '/content/diabetic-retinopathy-xai'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from xai.gradcam import GradCAM, build_model_for_xai
from xai.lime_wrapper import LIMEWrapper
from datasets.idrid_dataset import INFERENCE_TRANSFORM

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── All paths ────────────────────────────────────────────────────────────────
PROJECT_ROOT = '/content/drive/MyDrive/Projects/diabetic-retinopathy-xai'
ckpt_dir     = f'{PROJECT_ROOT}/results/checkpoints'
IMG_DIR      = '/content/drive/MyDrive/IDRiD/grading/images/test'
PRED_CSV     = f'{PROJECT_ROOT}/results/heatmaps/predictions.csv'
HEATMAP_DIR  = f'{PROJECT_ROOT}/results/heatmaps'


def resolve_checkpoint(ckpt_dir, pattern):
    """
    Finds the right checkpoint even when Drive has appended ' (1)', ' (2)',
    etc. from a duplicate upload/share (this has happened 3+ times on this
    project). Prefers the file WITHOUT a duplicate suffix; warns if more
    than one candidate exists so a stale duplicate doesn't get loaded silently.
    """
    matches = sorted(glob.glob(os.path.join(ckpt_dir, pattern)))
    if not matches:
        raise FileNotFoundError(
            f"No checkpoint matching '{pattern}' in {ckpt_dir}. "
            f"Check it landed in 'My Drive' and not 'Shared with me'."
        )
    clean = [m for m in matches if '(' not in os.path.basename(m)]
    chosen = clean[0] if clean else matches[0]
    if len(matches) > 1:
        print(f"  [checkpoint] {len(matches)} files match '{pattern}': "
              f"{[os.path.basename(m) for m in matches]} -> using {os.path.basename(chosen)}")
    return chosen


CKPT = {
    'efficientnetb4': resolve_checkpoint(ckpt_dir, 'efficientnet_b4_clean_best*.pth'),
    'resnet50':       resolve_checkpoint(ckpt_dir, 'resnet50_clean_best*.pth'),
}

# GradCAM target layers
GRADCAM_LAYER = {
    # FIX: 'conv_head' is a raw 1x1 conv that runs BEFORE its batchnorm (bn2).
    # Its un-normalized, uncalibrated activations are exactly what produced
    # the flat-heatmap-with-one-hot-corner-pixel bug. 'blocks.6' is the last
    # MBConv stage — its output already has BN + SE applied internally, so
    # GradCAM gets well-scaled activations. Verified against timm's
    # efficientnet_b4 module tree (stages are blocks.0 ... blocks.6).
    'efficientnetb4': 'blocks.6',
    'resnet50':       'layer4',
}

# Frozen test IDs
with open(os.path.join(REPO_DIR, 'test_image_ids.json')) as f:
    TEST_IDS = json.load(f)
print(f'{len(TEST_IDS)} test IDs: {TEST_IDS[0]} to {TEST_IDS[-1]}')

# Verify all paths exist
print('\nPath check:')
for name, path in [('effnet ckpt', CKPT['efficientnetb4']),
                    ('resnet ckpt', CKPT['resnet50']),
                    ('img_dir',     IMG_DIR),
                    ('pred_csv',    PRED_CSV)]:
    print(f'  {name}: {os.path.exists(path)}  ({path})')


In [ ]:
# Cell 4: Load Shreya's predictions.csv
# NOTE: the old "delete every .npy file" step that used to live here has
# been moved to Cell 4b below and is OFF by default. That step is what wiped
# the whole heatmaps folder (including the already-correct ResNet-50 results)
# last time it got re-run by accident — it should never be a routine part of
# this notebook again.

pred_df = pd.read_csv(PRED_CSV)
print(f'predictions.csv loaded: {pred_df.shape}')
print(pred_df.head())
print(f'\nModels in CSV: {pred_df["model"].unique()}')
print(f'Image IDs sample: {pred_df["image_id"].values[:3]}')


In [ ]:
# Cell 4b: OPTIONAL — wipe existing heatmaps before regenerating
# Leave this OFF unless you specifically mean to delete everything in
# HEATMAP_DIR. Defaults to False so a stray "Run All" can never wipe
# results again.

CONFIRM_DELETE_ALL_HEATMAPS = False  # <-- change to True only if you mean it

if CONFIRM_DELETE_ALL_HEATMAPS:
    old_files = glob.glob(f'{HEATMAP_DIR}/**/*.npy', recursive=True)
    for f in old_files:
        os.remove(f)
    print(f'Deleted {len(old_files)} .npy files')
else:
    existing = glob.glob(f'{HEATMAP_DIR}/**/*.npy', recursive=True)
    print(f'Skipped deletion (CONFIRM_DELETE_ALL_HEATMAPS is False). '
          f'{len(existing)} .npy files currently on disk.')


In [ ]:
# Cell 5: Helper Functions

def load_image(image_id, img_dir):
    """Converts IDRiD_55 -> IDRiD_055.jpg for loading; output contract stays IDRiD_55."""
    num   = int(image_id.replace('IDRiD_', ''))
    fname = f'IDRiD_{num:03d}.jpg'
    path  = os.path.join(img_dir, fname)
    return Image.open(path).convert('RGB')

def clean_image_id(image_id):
    """Ensures IDRiD_55 format — strips zero padding from IDRiD_055."""
    num = int(image_id.replace('IDRiD_', ''))
    return f'IDRiD_{num}'

def get_predicted_grade(pred_df, image_id, model_name):
    """Looks up predicted grade from Shreya's predictions.csv."""
    clean_id = clean_image_id(image_id)
    row = pred_df[
        (pred_df['image_id'] == clean_id) &
        (pred_df['model']    == model_name)
    ]
    if len(row) == 0:
        # Try zero-padded version
        num      = int(image_id.replace('IDRiD_', ''))
        padded   = f'IDRiD_{num:03d}'
        row      = pred_df[
            (pred_df['image_id'] == padded) &
            (pred_df['model']    == model_name)
        ]
    if len(row) == 0:
        raise ValueError(f'No prediction found for {image_id} / {model_name} in predictions.csv')
    return int(row['predicted_grade'].values[0])

def save_heatmap(heatmap, model_name, method, image_id, heatmap_dir):
    """Saves heatmap .npy with correct filename format."""
    clean_id = clean_image_id(image_id)
    out_dir  = os.path.join(heatmap_dir, model_name, method)
    os.makedirs(out_dir, exist_ok=True)
    fname    = f'{model_name}_{method}_{clean_id}.npy'
    out_path = os.path.join(out_dir, fname)
    np.save(out_path, heatmap)
    return fname

def verify_heatmap(heatmap, pil_img):
    """Checks heatmap meets output contract. Raises if not."""
    assert heatmap.ndim   == 2,               f'Expected 2D, got {heatmap.ndim}D'
    assert heatmap.dtype  == np.float32,      f'Expected float32, got {heatmap.dtype}'
    assert heatmap.shape  == (pil_img.size[1], pil_img.size[0]), \
        f'Shape mismatch: heatmap {heatmap.shape} vs image {pil_img.size[::-1]}'
    assert heatmap.min()  >= 0.0,             f'Min below 0: {heatmap.min()}'
    assert heatmap.max()  <= 1.0,             f'Max above 1: {heatmap.max()}'

print('Helper functions ready')


In [ ]:
# Cell 6: SAMPLE BATCH — 3 images only
# Run this first. This is also the cheapest way to confirm the new
# EfficientNet-B4 GradCAM layer ('blocks.6') actually fixed the flat-heatmap
# bug, before committing to the ~3 hour full run in Cell 7.
# Send the printed stats (and ideally a quick visual check) to Shreya for
# format validation before running Cell 7.

SAMPLE_IDS = TEST_IDS[:3]
print(f'Sample IDs: {SAMPLE_IDS}\n')

for model_name in ['efficientnetb4', 'resnet50']:
    print(f'=== {model_name.upper()} ===')
    model = build_model_for_xai(model_name, CKPT[model_name], DEVICE)

    # GradCAM
    cam = GradCAM(model, target_layer_name=GRADCAM_LAYER[model_name], device=DEVICE)

    for image_id in SAMPLE_IDS:
        try:
            pred_grade = get_predicted_grade(pred_df, image_id, model_name)
            pil_img = load_image(image_id, IMG_DIR)
            tensor     = INFERENCE_TRANSFORM(pil_img)

            heatmap, _ = cam.generate(tensor, pil_img, target_class=pred_grade)
            verify_heatmap(heatmap, pil_img)
            fname = save_heatmap(heatmap, model_name, 'gradcam', image_id, HEATMAP_DIR)
            print(f'  GradCAM {clean_image_id(image_id)}: shape={heatmap.shape} dtype={heatmap.dtype} '
                  f'min={heatmap.min():.3f} max={heatmap.max():.3f} argmax_at={np.unravel_index(heatmap.argmax(), heatmap.shape)} -> {fname}')

        except Exception as e:
            print(f'  GradCAM {image_id} FAILED: {e} — skipping')

    # LIME
    lime_exp = LIMEWrapper(model, device=DEVICE)

    for image_id in SAMPLE_IDS:
        try:
            pred_grade = get_predicted_grade(pred_df, image_id, model_name)
            pil_img = load_image(image_id, IMG_DIR)

            heatmap, _ = lime_exp.generate(pil_img, target_class=pred_grade)
            verify_heatmap(heatmap, pil_img)
            fname = save_heatmap(heatmap, model_name, 'lime', image_id, HEATMAP_DIR)
            print(f'  LIME    {clean_image_id(image_id)}: shape={heatmap.shape} dtype={heatmap.dtype} '
                  f'min={heatmap.min():.3f} max={heatmap.max():.3f} -> {fname}')

        except Exception as e:
            print(f'  LIME {image_id} FAILED: {e} — skipping')

    del model
    torch.cuda.empty_cache()

print('\nSample batch done — send this output (and a quick visual check, Cell 6b below) to Shreya for format validation')
print('Only run Cell 7 after she confirms format is correct')


In [ ]:
# Cell 6b: OPTIONAL — visual sanity check on the sample batch
# Confirms the EfficientNet-B4 hotspot now lands on retina tissue, not the
# black background corner. Run after Cell 6.

image_id = SAMPLE_IDS[0]
pil = load_image(image_id, IMG_DIR)
clean_id = clean_image_id(image_id)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(pil); axes[0].set_title('Original'); axes[0].axis('off')

for col, model_name in enumerate(['efficientnetb4', 'resnet50'], 1):
    hm_path = os.path.join(HEATMAP_DIR, model_name, 'gradcam', f'{model_name}_gradcam_{clean_id}.npy')
    hm   = np.load(hm_path)
    orig = np.array(pil.resize((hm.shape[1], hm.shape[0])))
    colored = (cm.jet(hm)[:, :, :3] * 255).astype(np.uint8)
    blend   = (0.5 * orig + 0.5 * colored).astype(np.uint8)
    axes[col].imshow(blend)
    axes[col].set_title(f'{model_name} GradCAM')
    axes[col].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Cell 7: FULL RUN — all 27 images x 2 models x 2 methods
# Only run after Shreya confirms sample batch format is correct.
# GradCAM is fast (~2 sec/image), LIME is slow (~2-3 min/image).
# Total estimated time: ~3 hours.
# Resume-safe: already-completed images are skipped, so a crash or
# disconnect doesn't lose progress.

def already_done(model_name, method, image_id, heatmap_dir):
    clean_id = clean_image_id(image_id)
    path     = os.path.join(heatmap_dir, model_name, method,
                            f'{model_name}_{method}_{clean_id}.npy')
    return os.path.exists(path)

for model_name in ['efficientnetb4', 'resnet50']:
    print(f'\n=== {model_name.upper()} ===')
    model = build_model_for_xai(model_name, CKPT[model_name], DEVICE)

    # GradCAM — all 27
    print('-- GradCAM --')
    cam = GradCAM(model, target_layer_name=GRADCAM_LAYER[model_name], device=DEVICE)

    for i, image_id in enumerate(TEST_IDS):
        if already_done(model_name, 'gradcam', image_id, HEATMAP_DIR):
            print(f'  [{i+1:02d}/27] {clean_image_id(image_id)} already done — skipping')
            continue
        try:
            pred_grade = get_predicted_grade(pred_df, image_id, model_name)
            pil_img = load_image(image_id, IMG_DIR)
            tensor     = INFERENCE_TRANSFORM(pil_img)
            heatmap, _ = cam.generate(tensor, pil_img, target_class=pred_grade)
            verify_heatmap(heatmap, pil_img)
            fname = save_heatmap(heatmap, model_name, 'gradcam', image_id, HEATMAP_DIR)
            print(f'  [{i+1:02d}/27] {clean_image_id(image_id)} pred={pred_grade} -> {fname}')
        except Exception as e:
            print(f'  [{i+1:02d}/27] {image_id} FAILED: {e} — skipping')

    # LIME — all 27
    print('\n-- LIME (slow ~2-3 min/image) --')
    lime_exp = LIMEWrapper(model, device=DEVICE)

    for i, image_id in enumerate(TEST_IDS):
        if already_done(model_name, 'lime', image_id, HEATMAP_DIR):
            print(f'  [{i+1:02d}/27] {clean_image_id(image_id)} already done — skipping')
            continue
        try:
            pred_grade = get_predicted_grade(pred_df, image_id, model_name)
            pil_img = load_image(image_id, IMG_DIR)
            heatmap, _ = lime_exp.generate(pil_img, target_class=pred_grade)
            verify_heatmap(heatmap, pil_img)
            fname = save_heatmap(heatmap, model_name, 'lime', image_id, HEATMAP_DIR)
            print(f'  [{i+1:02d}/27] {clean_image_id(image_id)} pred={pred_grade} -> {fname}')
        except Exception as e:
            print(f'  [{i+1:02d}/27] {image_id} FAILED: {e} — skipping')

    del model
    torch.cuda.empty_cache()

print('\nFull run complete!')


In [ ]:
# Cell 8: Final Verification
import glob

expected = 27
total    = 0
all_ok   = True

print('File counts:')
for model_name in ['efficientnetb4', 'resnet50']:
    for method in ['gradcam', 'lime']:
        folder = os.path.join(HEATMAP_DIR, model_name, method)
        files  = glob.glob(os.path.join(folder, '*.npy'))
        count  = len(files)
        total += count
        status = 'OK' if count == expected else f'MISSING {expected - count}'
        print(f'  {model_name}/{method}: {count}/{expected} [{status}]')
        if count != expected:
            all_ok = False
        for f in files[:2]:
            print(f'    sample: {os.path.basename(f)}')

print(f'\nTotal: {total}/108')
print('\nWeek 2 P1 COMPLETE!' if all_ok else '\nSome files missing — check Cell 7 output')
